<a href="https://colab.research.google.com/github/vijayrajeshr/Flipkart__GridLockHackathon2.0/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flipkart Gridlock Hackathon 2.0

In [8]:
!pip install catboost pygeohash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

print("1. Loading datasets...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

def engineer_catboost_features(df):
    df = df.copy()

    # 1. Spatial Coordinates
    df['Latitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notna(x) else 0)
    df['Longitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notna(x) else 0)

    # 2. Time Extraction
    df['hours'] = df['timestamp'].astype(str).apply(lambda x: int(x.split(':')[0]) if pd.notna(x) and ':' in str(x) else 0)
    df['mins'] = df['timestamp'].astype(str).apply(lambda x: int(x.split(':')[1]) if pd.notna(x) and ':' in str(x) else 0)

    # 3. Cyclical Time Waves
    minutes_in_day = df['hours'] * 60 + df['mins']
    df['time_sin'] = np.sin(2 * np.pi * minutes_in_day / 1440)
    df['time_cos'] = np.cos(2 * np.pi * minutes_in_day / 1440)
    df['day_of_week'] = df['day'] % 7

    # 4. THE SECRET WEAPON: Categorical Interactions
    # We force the model to look at the unique combination of location + time + day
    df['geo_hour'] = df['geohash'].astype(str) + "_" + df['hours'].astype(str)
    df['geo_day'] = df['geohash'].astype(str) + "_" + df['day_of_week'].astype(str)

    # 5. Missing Values
    # CatBoost handles NaNs perfectly for categories, they just need to be strings
    cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geo_hour', 'geo_day']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype(str)

    # Standard numerical fills
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
    df['NumberofLanes'] = df['NumberofLanes'].fillna(df['NumberofLanes'].mode()[0])

    return df

print("2. Engineering Advanced Interaction Features...")
train_df = engineer_catboost_features(train)
test_df = engineer_catboost_features(test)

# Define our features
features = [
    'geohash', 'Latitude', 'Longitude',
    'hours', 'mins', 'time_sin', 'time_cos', 'day_of_week',
    'RoadType', 'NumberofLanes', 'LargeVehicles',
    'Landmarks', 'Temperature', 'Weather',
    'geo_hour', 'geo_day'
]

cat_features = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geo_hour', 'geo_day']

X_train = train_df[features]
y_train = train_df['demand']
X_test = test_df[features]

print("3. Initializing God-Tier CatBoost Model on GPU...")
model = CatBoostRegressor(
    iterations=4000,           # Massive amount of trees to catch every edge case
    learning_rate=0.03,        # Slow and steady to ensure max precision
    depth=9,                   # Very deep trees to map the complex interactions
    eval_metric='R2',          # We are explicitly optimizing for the competition metric
    cat_features=cat_features, # Telling CatBoost exactly which columns are strings
    task_type='GPU',           # Using the Colab GPU for incredible speed
    random_seed=42,
    verbose=250                # Prints progress every 250 iterations
)

print("4. Training Model (This will take a few minutes)...")
model.fit(X_train, y_train)

print("5. Generating Final Predictions...")
preds = model.predict(X_test)

# Safety check: Traffic cannot be less than 0
preds = np.clip(preds, 0, None)

sub = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

filename = 'CATBOOST_95_submission.csv'
sub.to_csv(filename, index=False)
print(f"SUCCESS! Download '{filename}' from the Colab file browser and submit it!")

1. Loading datasets...
2. Engineering Advanced Interaction Features...
